# Kalibrowany estymator IPW

**Autor:** Maciej Beręsewicz

## Pakiety

In [ ]:
## pip install pandas numpy matplotlib statsmodels scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import fsolve
from scipy.special import expit

## Dane

In [ ]:
jvs = pd.read_csv('../data/jvs.csv')
admin = pd.read_csv('../data/admin.csv')
print(f'admin: {admin.shape}, jvs: {jvs.shape}')

## Kontekst

Ostatnim razem wykorzystywaliśmy pseudo-funkcję największej wiarygodności:

$$
\ell^*(\boldsymbol{\gamma}) = \sum_{i \in S_{A}} \log \left\{\frac{\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)}{1-\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)}\right\} + \sum_{i \in S_{B}} d_i^B \log \left\{1-\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)\right\}.
$$

Ograniczenia:

- nie zawsze mamy dostęp do danych jednostkowych -- dysponujemy wyłącznie wartościami globalnymi
- wagi z MLE nie gwarantują odtworzenia znanych wartości globalnych

Podejście GEE -- definiujemy:

$$
\boldsymbol{G}(\boldsymbol{\gamma})=\sum_{i \in S_A} \boldsymbol{h}\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)-\sum_{i \in S_B} d_i^B \pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right) \boldsymbol{h}\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right)
$$

Dla $\boldsymbol{h}(\boldsymbol{x}_i, \boldsymbol{\gamma}) = \boldsymbol{x}_i \pi(\boldsymbol{x}_i, \boldsymbol{\gamma})^{-1}$ otrzymujemy **skalibrowane** IPW:

$$
\boldsymbol{G}(\boldsymbol{\gamma}) = \sum_{i \in S_A} \frac{\boldsymbol{x}_i}{\pi\left(\boldsymbol{x}_i, \boldsymbol{\gamma}\right) }-\sum_{i \in S_B} d_i^B \boldsymbol{x}_i
$$

## Funkcje

In [ ]:
wspol = ['private', 'size', 'nace', 'region']

def fit_ps_mle(admin, jvs, formula_vars):
    """PS metodą MLE (pseudo-likelihood z wagami JVS)."""
    jvs_sub = jvs[formula_vars].copy()
    jvs_sub['source'] = 0; jvs_sub['weight'] = jvs['weight'].values
    admin_sub = admin[formula_vars].copy()
    admin_sub['source'] = 1; admin_sub['weight'] = 1.0
    combined = pd.concat([jvs_sub, admin_sub], ignore_index=True)
    combined_dum = pd.get_dummies(combined[formula_vars],
                                  columns=[v for v in formula_vars if v != 'private'],
                                  dtype=float, drop_first=True)
    X = sm.add_constant(combined_dum)
    y = combined['source']
    model = sm.GLM(y, X, family=sm.families.Binomial(),
                   freq_weights=combined['weight']).fit()
    ps_admin = model.predict(X)[y == 1].values
    return model, ps_admin, X.columns.tolist()

def fit_ps_gee(admin, jvs, formula_vars):
    """PS metodą GEE (kalibracja do sum z JVS)."""
    admin_dum = pd.get_dummies(admin[formula_vars],
                               columns=[v for v in formula_vars if v != 'private'],
                               dtype=float, drop_first=True)
    jvs_dum = pd.get_dummies(jvs[formula_vars],
                              columns=[v for v in formula_vars if v != 'private'],
                              dtype=float, drop_first=True)
    all_cols = sorted(set(admin_dum.columns) | set(jvs_dum.columns))
    admin_dum = admin_dum.reindex(columns=all_cols, fill_value=0)
    jvs_dum = jvs_dum.reindex(columns=all_cols, fill_value=0)
    X_admin = sm.add_constant(admin_dum).values
    X_jvs = sm.add_constant(jvs_dum).values
    w_jvs = jvs['weight'].values
    tau_x = (X_jvs * w_jvs[:, None]).sum(axis=0)
    model_mle, _, _ = fit_ps_mle(admin, jvs, formula_vars)
    gamma0 = model_mle.params.values
    if len(gamma0) != X_admin.shape[1]:
        gamma0 = np.zeros(X_admin.shape[1])
    def equations(gamma):
        pi = expit(X_admin @ gamma)
        return (X_admin / pi[:, None]).sum(axis=0) - tau_x
    gamma_hat = fsolve(equations, gamma0)
    ps_admin = 1 / (1 + np.exp(-X_admin @ gamma_hat))
    col_names = ['const'] + all_cols
    return gamma_hat, ps_admin, col_names

def fit_ps_gee_totals(admin, pop_totals, formula_vars):
    """GEE z wartościami globalnymi zamiast danych JVS."""
    admin_dum = pd.get_dummies(admin[formula_vars],
                               columns=[v for v in formula_vars if v != 'private'],
                               dtype=float, drop_first=True)
    X_admin = sm.add_constant(admin_dum).values
    col_names = ['const'] + sorted(admin_dum.columns.tolist())
    tau_x = np.array([pop_totals.get(c, 0) for c in col_names])
    gamma0 = np.zeros(X_admin.shape[1])
    def equations(gamma):
        pi = expit(X_admin @ gamma)
        return (X_admin / pi[:, None]).sum(axis=0) - tau_x
    gamma_hat = fsolve(equations, gamma0)
    ps_admin = 1 / (1 + np.exp(-X_admin @ gamma_hat))
    return gamma_hat, ps_admin, col_names

def ipw_mean(y, ps):
    return np.average(y, weights=1.0/ps)

def check_balance(admin, jvs, var, w):
    cats = sorted(admin[var].unique())
    rows = []
    for c in cats:
        rows.append({'Kategoria': c,
                     'CBOP (raw)': round((admin[var] == c).mean(), 4),
                     'CBOP (IPW)': round(np.average(admin[var] == c, weights=w), 4),
                     'JVS (ważone)': round(np.average(jvs[var] == c, weights=jvs['weight']), 4)})
    return pd.DataFrame(rows)

## Przykład 1: MLE vs GEE z jedną zmienną

$$P(R_A = 1 | \text{size})$$

### MLE

In [ ]:
model_mle, ps_mle, _ = fit_ps_mle(admin, jvs, ['size'])
w_mle = 1.0 / ps_mle
mu_mle = ipw_mean(admin['single_shift'], ps_mle)
print(f'IPW-MLE (single_shift): {mu_mle:.4f}')
model_mle.params.round(4)

### GEE

In [ ]:
gamma_gee, ps_gee, col_gee = fit_ps_gee(admin, jvs, ['size'])
w_gee = 1.0 / ps_gee
mu_gee = ipw_mean(admin['single_shift'], ps_gee)
print(f'IPW-GEE (single_shift): {mu_gee:.4f}')
pd.Series(gamma_gee, index=col_gee).round(4)

### Balans

In [ ]:
print('MLE:')
print(check_balance(admin, jvs, 'size', w_mle))
print('\nGEE:')
print(check_balance(admin, jvs, 'size', w_gee))

## Przykład 2: MLE vs GEE ze wszystkimi zmiennymi

$$P(R_A = 1 | \text{size, nace, region, private})$$

In [ ]:
model_mle2, ps_mle2, _ = fit_ps_mle(admin, jvs, wspol)
w_mle2 = 1.0 / ps_mle2
mu_mle2 = ipw_mean(admin['single_shift'], ps_mle2)
print(f'IPW-MLE: {mu_mle2:.4f}')

gamma_gee2, ps_gee2, col_gee2 = fit_ps_gee(admin, jvs, wspol)
w_gee2 = 1.0 / ps_gee2
mu_gee2 = ipw_mean(admin['single_shift'], ps_gee2)
print(f'IPW-GEE: {mu_gee2:.4f}')

In [ ]:
pd.DataFrame({
    'Metoda': ['MLE', 'GEE'],
    'Szacunek': [round(mu_mle2, 4), round(mu_gee2, 4)],
    'N_hat': [round(w_mle2.sum(), 0), round(w_gee2.sum(), 0)]
})

### Porównanie wag

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(w_mle2, w_gee2, alpha=0.3, s=10)
lims = [0, max(w_mle2.max(), w_gee2.max()) * 1.05]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Wagi IPW (MLE)'); ax.set_ylabel('Wagi IPW (GEE)')
ax.set_title('Porównanie wag MLE vs GEE')
plt.tight_layout(); plt.show()

In [ ]:
print('Wagi MLE:')
print(pd.Series(w_mle2).describe().round(2))
print('\nWagi GEE:')
print(pd.Series(w_gee2).describe().round(2))

### Balans (GEE zapewnia zgodność rozkładów)

In [ ]:
print('MLE:')
print(check_balance(admin, jvs, 'size', w_mle2))
print('\nGEE:')
print(check_balance(admin, jvs, 'size', w_gee2))

## Przykład 3: GEE z wartościami globalnymi

Załóżmy, że znamy wartości globalne dla zmiennej `size`:

In [ ]:
pop_totals = {'const': 51870, 'size_M': 13758, 'size_S': 29551}

In [ ]:
gamma_gee3, ps_gee3, col_gee3 = fit_ps_gee_totals(admin, pop_totals, ['size'])
w_gee3 = 1.0 / ps_gee3
mu_gee3 = ipw_mean(admin['single_shift'], ps_gee3)

pd.DataFrame({
    'Metoda': ['GEE (dane jednostkowe)', 'GEE (wartości globalne)'],
    'Szacunek': [round(mu_gee, 4), round(mu_gee3, 4)]
})